In [ ]:
%%sql -r dataframe_1
Use database social_media_floodgates;

select * from TWEET_INGEST



In [ ]:
//simple select statements -- are you seeing 9 rows?
select raw_status
from social_media_floodgates.public.tweet_ingest;

In [ ]:
%%sql -r entities_result
select raw_status:entities
from social_media_floodgates.public.tweet_ingest;

In [ ]:
%%sql -r hashtags_result
select raw_status:entities:hashtags
from social_media_floodgates.public.tweet_ingest;

In [ ]:
%%sql -r first_hashtag
//This query returns just the first hashtag in each tweet
select raw_status:entities:hashtags[0].text
from social_media_floodgates.public.tweet_ingest;

In [ ]:
%%sql -r non_null_hashtags
//Get rid of any tweet that doesn't include any hashtags
select raw_status:entities:hashtags[0].text
from social_media_floodgates.public.tweet_ingest
where raw_status:entities:hashtags[0].text is not null;

In [ ]:
%%sql -r tweets_by_date
//Perform a simple CAST on the created_at key
//Add an ORDER BY clause to sort by the tweet's creation date
select raw_status:created_at::date
from social_media_floodgates.public.tweet_ingest
order by raw_status:created_at::date;

In [ ]:
//Flatten statements can return nested entities only (and ignore the higher level objects)
select value
from tweet_ingest
,lateral flatten
(input => raw_status:entities:urls);


In [ ]:

select value
from tweet_ingest
,table(flatten(raw_status:entities:urls));

In [ ]:
//Flatten and return just the hashtag text, CAST the text as VARCHAR
select value:text::varchar as hashtag_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:hashtags);

//Add the Tweet ID and User ID to the returned table so we could join the hashtag back to it's source tweet
select raw_status:user:name::text as user_name
,raw_status:id as tweet_id
,value:text::varchar as hashtag_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:hashtags);

In [ ]:
create or replace view social_media_floodgates.public.urls_normalized as
(select raw_status:user:name::text as user_name
,raw_status:id as tweet_id
,value:display_url::text as url_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:urls)
);

In [ ]:
%%sql -r dataframe_7
select * from social_media_floodgates.public.urls_normalized